# CRAFT Stage 1: SPLADE Sparse Retrieval

This notebook implements Stage 1 of the CRAFT pipeline: SPLADE-based sparse retrieval for initial candidate filtering.

**Pipeline Flow:**
1. Load table metadata and enrich with Gemini-generated summaries
2. Apply SPLADE encoding to create sparse representations
3. Filter large corpus (169K-419K tables) down to 5,000 candidates per query
4. Output initial rankings for Stage 2 processing

In [ ]:
# Import CRAFT modular components
from craft_core import CRAFTConfig, DataLoader, TableProcessor, ResultsManager, setup_logging
from craft_stages import create_splade_retriever, setup_nq_pipeline, setup_ottqa_pipeline

import pandas as pd
import numpy as np
from tqdm import tqdm
import json
import logging

# Setup logging
setup_logging("INFO")
logger = logging.getLogger(__name__)

# Configuration
DATASET = "nq"  # "nq" or "ottqa"

# Initialize CRAFT configuration
config = CRAFTConfig(DATASET)
data_loader = DataLoader()
table_processor = TableProcessor()
results_manager = ResultsManager()

print(f"🔍 CRAFT Stage 1: SPLADE Sparse Retrieval")
print(f"📊 Dataset: {DATASET.upper()}")
print(f"🎯 Target candidates: {config.stage1_candidates}")

print(f"Initializing SPLADE Stage 1 for {DATASET.upper()} dataset")
print(f"Transformers version: {transformers.__version__}")

In [ ]:
# Load data using modular components
logger.info("📂 Loading base datasets...")

# Load metadata and questions
metadata = data_loader.load_metadata(config.get_path("metadata"))
questions = data_loader.load_questions(config.get_path("questions"))

# Load table summaries if available
summary_path = config.base_dir / "datasets" / f"{DATASET}_table_summary_table_description.jsonl"
summaries = {}

if summary_path.exists():
    logger.info(f"📋 Loading table summaries from {summary_path}")
    with open(summary_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            summaries[data['table_id']] = data.get('summary', '')
    
    # Add summaries to metadata
    metadata['summary'] = metadata['TableID'].map(summaries).fillna('')
    logger.info(f"✅ Added summaries for {len(summaries)} tables")
else:
    logger.warning("⚠️ No summary file found, using metadata only")
    metadata['summary'] = ''

logger.info(f"📊 Loaded {len(metadata)} tables and {len(questions)} questions")

In [ ]:
# Initialize SPLADE retriever using modular components
logger.info("🤖 Initializing SPLADE retriever...")

# Get model configuration for dataset
pipeline_config = setup_nq_pipeline() if DATASET == "nq" else setup_ottqa_pipeline()
model_name = pipeline_config['stage1']

# Create retriever
retriever = create_splade_retriever(model_name=model_name, device="cpu")
logger.info(f"✅ SPLADE retriever initialized with model: {model_name}")

In [ ]:
# Create table corpus using table processor
logger.info("📝 Creating table text corpus...")

table_texts = []
table_ids = []

for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc="Processing tables"):
    # Create table text representation with summaries
    table_text = table_processor.create_table_text(row, include_summary=True)
    
    table_texts.append(table_text)
    table_ids.append(row['TableID'])

logger.info(f"✅ Created corpus with {len(table_texts)} tables")
logger.info(f"📏 Average table text length: {np.mean([len(text) for text in table_texts]):.1f} characters")

In [ ]:
# Build SPLADE index
logger.info("🏗️ Building SPLADE index...")
retriever.build_index(table_texts, table_ids)
logger.info("✅ SPLADE index built successfully")

In [ ]:
# Process queries and retrieve candidates
logger.info(f"🔍 Processing {len(questions)} queries...")

all_results = {}
output_lines = []

for idx, (_, query_row) in enumerate(tqdm(questions.iterrows(), desc="Retrieving candidates")):
    # Extract query information
    query_id = str(idx)
    if 'query_id' in query_row:
        query_id = str(query_row['query_id'])
    elif 'id' in query_row:
        query_id = str(query_row['id'])
    
    question = query_row.get('question', query_row.get('text', ''))
    gold_table_id = query_row.get('gold_table_id', '')
    
    # Retrieve top candidates
    candidates = retriever.retrieve(question, top_k=config.stage1_candidates)
    all_results[query_id] = candidates
    
    # Format for output file
    formatted_result = results_manager.format_retrieval_results(
        {tid: score for tid, score in candidates},
        query_id,
        question,
        gold_table_id if gold_table_id else None
    )
    output_lines.append(formatted_result)

logger.info(f"✅ Processed all queries, retrieved {config.stage1_candidates} candidates each")

In [ ]:
# Save results using results manager
logger.info("💾 Saving Stage 1 results...")

# Save pickle file
pickle_path = results_manager.save_stage_results(
    all_results, 1, DATASET, config.get_path("stage1_output").parent
)

# Save text format for compatibility
text_output_path = config.get_path("stage1_output")
text_output_path.parent.mkdir(parents=True, exist_ok=True)

with open(text_output_path, 'w') as f:
    for line in output_lines:
        f.write(line + '\n')

# Display summary statistics
if 'gold_table_id' in questions.columns:
    # Compute basic retrieval metrics
    from craft_core import EvaluationMetrics
    metrics = EvaluationMetrics()
    
    gold_answers = dict(zip(
        [str(i) for i in range(len(questions))],
        questions['gold_table_id']
    ))
    
    rankings = {qid: [tid for tid, _ in results] for qid, results in all_results.items()}
    recall_scores = metrics.compute_recall_at_k(rankings, gold_answers, [1, 10, 50])
    
    logger.info("📊 Stage 1 Retrieval Metrics:")
    for k, score in recall_scores.items():
        logger.info(f"   Recall@{k}: {score:.2f}%")

logger.info(f"✅ Stage 1 complete!")
logger.info(f"📁 Results saved to:")
logger.info(f"   Pickle: {pickle_path}")
logger.info(f"   Text: {text_output_path}")
logger.info(f"🎯 Ready for Stage 2 processing")

In [ ]:
# Table text generation function
def setTableText(metadata, setting):
    """Generate table text representations with different combinations of fields."""
    
    if setting == 10:  # Title + Headers + CellValues
        metadata['table_text'] = metadata[['Table_Title','Table_Headers','Table_CellValues']].astype(str).agg(' | '.join, axis=1)
    elif setting == 6 and 'Description' in metadata.columns:
        metadata['table_text'] = metadata[['Table_Title','Table_Headers','Description']].astype(str).agg(' | '.join, axis=1)
    else:
        # Fallback to title + headers
        metadata['table_text'] = metadata[['Table_Title','Table_Headers']].astype(str).agg(' | '.join, axis=1)
    
    return metadata

# Apply table text generation
setting = 10
metadata = setTableText(metadata, setting)

# Create corpus for SPLADE
corpus = [text.lower() for text in metadata["table_text"].iloc[1:]]  # Skip header
corpus_ids = [text for text in metadata["TableID"].iloc[1:]]

print(f"Setting {setting}: {len(corpus)} tables processed")
print(f"Sample text: {corpus[0][:200]}...")